# Assay Plotter: Professional Dose-Response Analysis

This notebook generates high-resolution line-and-scatter plots for biochemical assays (e.g., Inhibition, Activity, Concentration). 

### Features:
- **Smooth Curves**: Uses 4-Parameter Logistic (4PL) regression or Spline interpolation for smooth data lines.
- **IC50 Analysis**: Automatically calculates and visualizes IC50 values.
- **Multi-Format Support**: Handles both standard vertical layouts and compound-based horizontal layouts.
- **Professional Styling**: Uses a high-quality scientific plotting style with a safe color palette (no red, maroon, or black).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import os
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline

def logistic4(x, a, b, c, d):
    """4-Parameter Logistic Curve for IC50 fitting."""
    return d + (a - d) / (1 + (x / c)**b)

def calculate_ic50(x_data, y_data):
    """Calculates IC50 using 4PL regression."""
    try:
        p0 = [min(y_data), 1, np.median(x_data), max(y_data)]
        popt, _ = curve_fit(logistic4, x_data, y_data, p0=p0, maxfev=20000)
        a, b, c, d = popt
        if (a-d) != 0 and (50-d)/(a-d) > 0:
            val = (a - d) / (50 - d) - 1
            if val > 0:
                return c * (val**(1/b)), popt
        return None, None
    except:
        return None, None

### 1. Data Loading
Upload your `.csv` file and specify the path below.

In [ ]:
# Path to your CSV file
csv_path = 'assay_results.csv'  # or 'HRBC Job.csv'

try:
    try:
        df = pd.read_csv(csv_path)
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding='latin-1')
    
    # CLEANING: Remove any leading/trailing whitespace from column names
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all').dropna(axis=1, how='all')
    
    print(f"Successfully loaded: {csv_path}")
    display(df.head())
    
except FileNotFoundError:
    print(f"Error: {csv_path} not found. Please ensure the file is in the same directory.")
except Exception as e:
    print(f"An error occurred: {e}")

### 2. Visualization and IC50 Analysis

In [ ]:
try:
    # Custom palette avoiding restricted colors
    custom_colors = ['#FF8C00', '#00CED1', '#FF1493', '#000000', '#FF4500', '#008B8B', '#2563eb', '#16a34a']
    sns.set_theme(style="whitegrid", font_scale=1.2)
    fig, ax = plt.subplots(figsize=(12, 8))

    ic50_results = []
    
    if 'Compound' in df.columns:
        # Horizontal Format
        value_cols = [c for c in df.columns if c != 'Compound']
        x_numeric = np.array([float(re.search(r'(\d+)', c).group(1)) if re.search(r'(\d+)', c) else c for c in value_cols])
        x_smooth = np.linspace(min(x_numeric), max(x_numeric), 300)
        
        for i, (idx, row) in enumerate(df.iterrows()):
            name = str(row['Compound']).strip()
            if not name or name == 'nan': continue
            y_values = row[value_cols].values.astype(float)
            color = custom_colors[i % len(custom_colors)]
            
            ic50, popt = calculate_ic50(x_numeric, y_values)
            ax.scatter(x_numeric, y_values, color=color, s=80, edgecolors='white', zorder=5)

            if popt is not None:
                ax.plot(x_smooth, logistic4(x_smooth, *popt), color=color, label=name, linewidth=2.5, zorder=4)
                if ic50 and min(x_numeric) <= ic50 <= max(x_numeric): 
                    ic50_results.append((name, ic50, color))
            else:
                try:
                    spline = make_interp_spline(x_numeric, y_values, k=2)
                    ax.plot(x_smooth, spline(x_smooth), color=color, label=name, linewidth=2, alpha=0.6, zorder=4)
                except:
                    ax.plot(x_numeric, y_values, color=color, label=name, linewidth=2, alpha=0.6, zorder=4)
        plt.xlabel('Concentration', fontweight='bold')
        
    else:
        # Vertical Format
        col_x = 'Mass (ug)' if 'Mass (ug)' in df.columns else df.columns[0]
        x_data = df[col_x].values
        x_smooth = np.linspace(min(x_data), max(x_data), 300)
        data_cols = [c for c in df.columns if c != col_x]
        
        for i, col in enumerate(data_cols):
            y_data = df[col].values
            color = custom_colors[i % len(custom_colors)]
            ic50, popt = calculate_ic50(x_data, y_data)
            ax.scatter(x_data, y_data, color=color, s=80, edgecolors='white', zorder=5)
            
            if popt is not None:
                ax.plot(x_smooth, logistic4(x_smooth, *popt), color=color, label=col, linewidth=2.5, zorder=4)
                if ic50 and min(x_data) <= ic50 <= max(x_data): 
                    ic50_results.append((col, ic50, color))
            else:
                try:
                    spline = make_interp_spline(x_data, y_data, k=2)
                    ax.plot(x_smooth, spline(x_smooth), color=color, label=col, linewidth=2, alpha=0.6, zorder=4)
                except:
                    ax.plot(x_data, y_data, color=color, label=col, linewidth=2, alpha=0.6, zorder=4)
        plt.xlabel(f'Concentration ({col_x})', fontweight='bold')

    # IC50 REFERENCE LINES
    ax.axhline(50, color='#2F4F4F', linestyle=':', linewidth=1.2, alpha=0.5, zorder=1)
    for name, ic50, color in ic50_results:
        ax.vlines(x=ic50, ymin=-10, ymax=50, colors=color, linestyles=':', linewidth=1.5, alpha=0.7, zorder=6)
        ax.plot(ic50, 50, marker='o', color='white', markeredgecolor=color, markersize=8, markeredgewidth=1.5, zorder=7)
        ax.text(ic50, -8, f'{ic50:.1f}', color=color, fontweight='bold', ha='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # Summary Panel
    if ic50_results:
        summary = "IC50 Values:\n" + "\n".join([f"{n}: {ic:.2f}" for n, ic, c in ic50_results])
        ax.text(0.98, 0.02, summary, transform=ax.transAxes, fontsize=11, fontweight='bold', 
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='darkgrey'), 
                verticalalignment='bottom', horizontalalignment='right', family='monospace')

    plt.ylabel('% Inhibition', fontweight='bold')
    plt.title('Assay Dose-Response Analysis', fontweight='bold', pad=25, fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True, borderpad=1)
    plt.ylim(-10, 110)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Visualization error: {e}")